In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/README.md
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/labels.txt
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/gitattributes
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/mapping_emotions.json
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-less_small/dev.jsonl
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-less_small/test.jsonl
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-less_small/train.jsonl
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-full_small/dev.jsonl
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-full_small/test.jsonl
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-full_small/train.jsonl
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-less/dev.jsonl
/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-less/test.jsonl
/kaggle/

In [5]:
# ============================================================
# MODEL 1: Utterance-only RoBERTa Baseline for EmoPillars
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ============================================================
# 1. Configuration
# ============================================================

class CFG:
    DATA_ROOT = Path("/kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset")
    
    # Start with small dataset to confirm everything works.
    # After successful run, set USE_SMALL = False.
    USE_SMALL = False
    
    MODEL_NAME = "roberta-base"
    
    MAX_LEN = 128
    BATCH_SIZE = 16
    EPOCHS = 3
    LR = 2e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.10
    
    # In EmoPillars, labels have expressiveness scores.
    # We keep labels whose score >= 0.30.
    LABEL_THRESHOLD = 0.30
    
    # Prediction threshold will be tuned on dev set later.
    DEFAULT_PRED_THRESHOLD = 0.30
    
    SEED = 42
    
    OUTPUT_DIR = Path("/kaggle/working/roberta_utterance_only_model")


CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if CFG.USE_SMALL:
    TRAIN_FILE = CFG.DATA_ROOT / "context-less_small/train.jsonl"
    DEV_FILE   = CFG.DATA_ROOT / "context-less_small/dev.jsonl"
    TEST_FILE  = CFG.DATA_ROOT / "context-less_small/test.jsonl"
else:
    TRAIN_FILE = CFG.DATA_ROOT / "context-less/train.jsonl"
    DEV_FILE   = CFG.DATA_ROOT / "context-less/dev.jsonl"
    TEST_FILE  = CFG.DATA_ROOT / "context-less/test.jsonl"

LABEL_FILE = CFG.DATA_ROOT / "labels.txt"

print("Train file:", TRAIN_FILE)
print("Dev file:", DEV_FILE)
print("Test file:", TEST_FILE)
print("Output dir:", CFG.OUTPUT_DIR)


# ============================================================
# 2. Reproducibility
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 3. Load Emotion Labels
# ============================================================

def load_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f if line.strip()]
    return labels


LABELS = load_labels(LABEL_FILE)
NUM_LABELS = len(LABELS)

id2label = {i: label for i, label in enumerate(LABELS)}
label2id = {label: i for i, label in enumerate(LABELS)}

print("Number of labels:", NUM_LABELS)
print(LABELS)


# ============================================================
# 4. Inspect One Record
# ============================================================

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("\nSample keys:")
print(sample.keys())

print("\nSample record:")
print(json.dumps(sample, indent=2, ensure_ascii=False)[:2500])


# ============================================================
# 5. Convert Labels to Multi-hot Vector
# ============================================================

def make_multihot(label_indices, expressiveness, num_labels, threshold=0.30):
    """
    label_indices example: ["22", "25", "24", "2"]
    expressiveness example: [1.0, 0.8, 0.9, 0.4]
    
    Output: 28-dimensional multi-hot vector.
    """
    
    y = np.zeros(num_labels, dtype=np.float32)
    
    if label_indices is None:
        return y
    
    if expressiveness is None:
        expressiveness = [1.0] * len(label_indices)
    
    for label_idx, score in zip(label_indices, expressiveness):
        try:
            idx = int(label_idx)
            score = float(score)
        except:
            continue
        
        if 0 <= idx < num_labels and score >= threshold:
            y[idx] = 1.0
    
    return y


test_y = make_multihot(
    sample["labels"],
    sample["expressiveness"],
    NUM_LABELS,
    threshold=CFG.LABEL_THRESHOLD
)

print("\nPositive labels in sample:")
print([LABELS[i] for i, v in enumerate(test_y) if v == 1])


# ============================================================
# 6. Load JSONL Files
# ============================================================

def load_emopillars_contextless(path, num_labels, threshold=0.30):
    rows = []
    
    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Loading {path.name}"):
            line = line.strip()
            if not line:
                continue
            
            record = json.loads(line)
            
            utterance = record.get("utterance", "")
            label_indices = record.get("labels", [])
            expressiveness = record.get("expressiveness", [])
            
            y = make_multihot(
                label_indices,
                expressiveness,
                num_labels,
                threshold=threshold
            )
            
            if len(utterance.strip()) == 0:
                continue
            
            if y.sum() == 0:
                continue
            
            rows.append({
                "utterance": utterance,
                "labels": y,
                "primary_emotion": record.get("primary_emotion", None),
                "plot_id": record.get("plot_id", None)
            })
    
    df = pd.DataFrame(rows)
    return df


train_df = load_emopillars_contextless(TRAIN_FILE, NUM_LABELS, CFG.LABEL_THRESHOLD)
dev_df   = load_emopillars_contextless(DEV_FILE, NUM_LABELS, CFG.LABEL_THRESHOLD)
test_df  = load_emopillars_contextless(TEST_FILE, NUM_LABELS, CFG.LABEL_THRESHOLD)

print("\nTrain shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Test shape:", test_df.shape)

print("\nProcessed sample:")
print(train_df.iloc[0]["utterance"])
print([LABELS[i] for i, v in enumerate(train_df.iloc[0]["labels"]) if v == 1])


# ============================================================
# 7. Label Distribution
# ============================================================

def label_distribution(df, labels):
    label_matrix = np.stack(df["labels"].values)
    counts = label_matrix.sum(axis=0)
    
    dist = pd.DataFrame({
        "label": labels,
        "count": counts.astype(int)
    }).sort_values("count", ascending=False)
    
    return dist


dist_df = label_distribution(train_df, LABELS)

print("\nTraining label distribution:")
display(dist_df)


# ============================================================
# 8. Dataset and DataLoader
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)


class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df["utterance"].tolist()
        self.labels = np.stack(df["labels"].values)
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = self.labels[idx]
        
        encoded = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.float)
        }


train_dataset = EmotionDataset(train_df, tokenizer, CFG.MAX_LEN)
dev_dataset   = EmotionDataset(dev_df, tokenizer, CFG.MAX_LEN)
test_dataset  = EmotionDataset(test_df, tokenizer, CFG.MAX_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# 9. Model
# ============================================================

class RobertaEmotionClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.2):
        super().__init__()
        
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        cls_rep = outputs.last_hidden_state[:, 0, :]
        cls_rep = self.dropout(cls_rep)
        logits = self.classifier(cls_rep)
        
        return logits


model = RobertaEmotionClassifier(
    model_name=CFG.MODEL_NAME,
    num_labels=NUM_LABELS,
    dropout=0.2
)

model = model.to(device)


# ============================================================
# 10. Loss, Optimizer, Scheduler
# ============================================================

criterion = nn.BCEWithLogitsLoss()

optimizer = AdamW(
    model.parameters(),
    lr=CFG.LR,
    weight_decay=CFG.WEIGHT_DECAY
)

total_steps = len(train_loader) * CFG.EPOCHS
warmup_steps = int(total_steps * CFG.WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print("Total steps:", total_steps)
print("Warmup steps:", warmup_steps)


# ============================================================
# 11. Metrics
# ============================================================

def compute_metrics(y_true, y_prob, threshold=0.30):
    y_pred = (y_prob >= threshold).astype(int)
    
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    
    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall
    }


def find_best_threshold(y_true, y_prob):
    best_t = 0.30
    best_f1 = -1
    
    for t in np.arange(0.05, 0.96, 0.01):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    
    return float(best_t), float(best_f1)


# ============================================================
# 12. Training and Evaluation Functions
# ============================================================

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    
    progress = tqdm(loader, desc="Training", leave=False)
    
    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        progress.set_postfix({"loss": loss.item()})
    
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.30):
    model.eval()
    
    total_loss = 0
    all_probs = []
    all_labels = []
    
    progress = tqdm(loader, desc="Evaluating", leave=False)
    
    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        
        probs = torch.sigmoid(logits)
        
        total_loss += loss.item()
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    
    y_prob = np.vstack(all_probs)
    y_true = np.vstack(all_labels)
    
    metrics = compute_metrics(y_true, y_prob, threshold=threshold)
    
    return total_loss / len(loader), metrics, y_true, y_prob


# ============================================================
# 13. Training Loop
# ============================================================

best_macro_f1 = -1
best_model_path = CFG.OUTPUT_DIR / "best_model.pt"

history = []

for epoch in range(1, CFG.EPOCHS + 1):
    print(f"\n================ Epoch {epoch}/{CFG.EPOCHS} ================")
    
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device
    )
    
    dev_loss, dev_metrics, dev_y_true, dev_y_prob = evaluate(
        model=model,
        loader=dev_loader,
        criterion=criterion,
        device=device,
        threshold=CFG.DEFAULT_PRED_THRESHOLD
    )
    
    print(f"Train loss: {train_loss:.4f}")
    print(f"Dev loss:   {dev_loss:.4f}")
    
    for k, v in dev_metrics.items():
        print(f"{k}: {v:.4f}")
    
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "dev_loss": dev_loss,
        **dev_metrics
    }
    history.append(row)
    
    if dev_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = dev_metrics["macro_f1"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved best model: {best_model_path}")
        print(f"Best dev macro F1: {best_macro_f1:.4f}")


history_df = pd.DataFrame(history)
history_df.to_csv(CFG.OUTPUT_DIR / "training_history.csv", index=False)

print("\nTraining finished.")
display(history_df)


# ============================================================
# 14. Tune Best Threshold on Dev Set
# ============================================================

model.load_state_dict(torch.load(best_model_path, map_location=device))
model = model.to(device)

dev_loss, dev_metrics, dev_y_true, dev_y_prob = evaluate(
    model=model,
    loader=dev_loader,
    criterion=criterion,
    device=device,
    threshold=CFG.DEFAULT_PRED_THRESHOLD
)

best_threshold, best_dev_f1 = find_best_threshold(dev_y_true, dev_y_prob)

print("\nBest threshold:", best_threshold)
print("Best dev macro F1:", best_dev_f1)

with open(CFG.OUTPUT_DIR / "best_threshold.txt", "w") as f:
    f.write(str(best_threshold))


# ============================================================
# 15. Final Test Evaluation
# ============================================================

test_loss, test_metrics_default, test_y_true, test_y_prob = evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    threshold=CFG.DEFAULT_PRED_THRESHOLD
)

test_metrics_best = compute_metrics(
    test_y_true,
    test_y_prob,
    threshold=best_threshold
)

print("\n================ FINAL TEST RESULTS ================")
print(f"Test loss: {test_loss:.4f}")

print("\nUsing default threshold:", CFG.DEFAULT_PRED_THRESHOLD)
for k, v in test_metrics_default.items():
    print(f"{k}: {v:.4f}")

print("\nUsing best dev threshold:", round(best_threshold, 3))
for k, v in test_metrics_best.items():
    print(f"{k}: {v:.4f}")


result_df = pd.DataFrame([
    {
        "threshold_type": "default",
        "threshold": CFG.DEFAULT_PRED_THRESHOLD,
        **test_metrics_default
    },
    {
        "threshold_type": "best_dev",
        "threshold": best_threshold,
        **test_metrics_best
    }
])

result_df.to_csv(CFG.OUTPUT_DIR / "test_results.csv", index=False)
display(result_df)


# ============================================================
# 16. Per-Class Classification Report
# ============================================================

test_y_pred = (test_y_prob >= best_threshold).astype(int)

report = classification_report(
    test_y_true,
    test_y_pred,
    target_names=LABELS,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()
report_df.to_csv(CFG.OUTPUT_DIR / "classification_report.csv")

display(report_df)


# ============================================================
# 17. Save Tokenizer and Config
# ============================================================

tokenizer.save_pretrained(CFG.OUTPUT_DIR)

config = {
    "model_name": CFG.MODEL_NAME,
    "num_labels": NUM_LABELS,
    "labels": LABELS,
    "label2id": label2id,
    "id2label": id2label,
    "max_len": CFG.MAX_LEN,
    "best_threshold": best_threshold,
    "use_small": CFG.USE_SMALL
}

with open(CFG.OUTPUT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("\nSaved files:")
print(os.listdir(CFG.OUTPUT_DIR))


# ============================================================
# 18. Inference Function
# ============================================================

@torch.no_grad()
def predict_emotions(text, model, tokenizer, labels, threshold=None, top_k=8):
    model.eval()
    
    if threshold is None:
        threshold = best_threshold
    
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        max_length=CFG.MAX_LEN,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )
    
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    
    logits = model(input_ids=input_ids, attention_mask=attention_mask)
    probs = torch.sigmoid(logits).cpu().numpy()[0]
    
    selected = []
    for i, p in enumerate(probs):
        if p >= threshold:
            selected.append({
                "emotion": labels[i],
                "score": float(p)
            })
    
    selected = sorted(selected, key=lambda x: x["score"], reverse=True)
    
    top_predictions = sorted(
        [
            {
                "emotion": labels[i],
                "score": float(probs[i])
            }
            for i in range(len(labels))
        ],
        key=lambda x: x["score"],
        reverse=True
    )[:top_k]
    
    return {
        "selected_emotions": selected,
        "top_predictions": top_predictions
    }


example_text = "I cannot believe this finally happened. I am so happy right now!"

prediction = predict_emotions(
    text=example_text,
    model=model,
    tokenizer=tokenizer,
    labels=LABELS,
    threshold=best_threshold,
    top_k=8
)

print("\nExample text:")
print(example_text)

print("\nSelected emotions:")
print(prediction["selected_emotions"])

print("\nTop predictions:")
print(prediction["top_predictions"])


# ============================================================
# 19. Save Test Predictions
# ============================================================

def create_prediction_df(df, y_prob, threshold, labels):
    rows = []
    
    for i in range(len(df)):
        probs = y_prob[i]
        pred_indices = np.where(probs >= threshold)[0]
        
        pred_labels = [labels[j] for j in pred_indices]
        pred_scores = [float(probs[j]) for j in pred_indices]
        
        rows.append({
            "utterance": df.iloc[i]["utterance"],
            "true_primary_emotion": df.iloc[i]["primary_emotion"],
            "predicted_labels": pred_labels,
            "predicted_scores": pred_scores
        })
    
    return pd.DataFrame(rows)


pred_df = create_prediction_df(
    test_df,
    test_y_prob,
    best_threshold,
    LABELS
)

pred_path = CFG.OUTPUT_DIR / "test_predictions.csv"
pred_df.to_csv(pred_path, index=False)

print("\nSaved predictions to:", pred_path)
display(pred_df.head(10))

Train file: /kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-less/train.jsonl
Dev file: /kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-less/dev.jsonl
Test file: /kaggle/input/datasets/muhammadsaad003/cse495b/NLP dataset/context-less/test.jsonl
Output dir: /kaggle/working/roberta_utterance_only_model
Device: cuda
GPU: Tesla T4
Number of labels: 28
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']

Sample keys:
dict_keys(['plot_id', 'utterance', 'labels', 'expressiveness', 'primary_emotion', 'all_emotions', 'all_emotions_mapped', 'emotions_used_to_generate_context', 'raw_emotion_explication'])

Sample record:
{
  "plot_id": 356,
  "utterance": "He's right. We have become 

Loading train.jsonl: 0it [00:00, ?it/s]

Loading dev.jsonl: 0it [00:00, ?it/s]

Loading test.jsonl: 0it [00:00, ?it/s]


Train shape: (266456, 4)
Dev shape: (33298, 4)
Test shape: (33302, 4)

Processed sample:
He's right. We have become heartless. We need to remember the humanity we've lost.
['anger', 'realization', 'remorse', 'sadness']

Training label distribution:


...


================ FINAL TEST RESULTS ================
Test loss: 0.0949

Using default threshold: 0.3
macro_f1: 0.7919
micro_f1: 0.8229
macro_precision: 0.7562
macro_recall: 0.8341

Using best dev threshold: 0.39
macro_f1: 0.7970
micro_f1: 0.8286
macro_precision: 0.7943
macro_recall: 0.8030


,threshold_type,threshold,macro_f1,micro_f1,macro_precision,macro_recall
0,default,0.30,0.791922,0.822889,0.756210,0.834064
1,best_dev,0.39,0.797004,0.828630,0.794308,0.802967


,precision,recall,f1-score,support
admiration,0.783534,0.805252,0.794245,4113.0
amusement,0.849953,0.726693,0.783505,1255.0
anger,0.872116,0.917164,0.894073,7376.0
annoyance,0.840000,0.862203,0.850957,5675.0
approval,0.589135,0.566593,0.577644,1359.0
caring,0.730240,0.780171,0.754380,2925.0
confusion,0.804082,0.844496,0.823793,8257.0
curiosity,0.865856,0.871324,0.868582,5067.0
desire,0.824715,0.869525,0.846528,5411.0
disappointment,0.836962,0.890367,0.862839,8118.0



Saved files:
['config.json', 'classification_report.csv', 'best_model.pt', 'tokenizer.json', 'training_history.csv', 'tokenizer_config.json', 'test_predictions.csv', 'test_results.csv', 'best_threshold.txt']

Example text:
I cannot believe this finally happened. I am so happy right now!

Selected emotions:
[{'emotion': 'joy', 'score': 0.9972164630889893}, {'emotion': 'surprise', 'score': 0.956759512424469}, {'emotion': 'excitement', 'score': 0.45547598600387573}]

Top predictions:
[{'emotion': 'joy', 'score': 0.9972164630889893}, {'emotion': 'surprise', 'score': 0.956759512424469}, {'emotion': 'excitement', 'score': 0.45547598600387573}, {'emotion': 'relief', 'score': 0.31421202421188354}, {'emotion': 'admiration', 'score': 0.07041355222463608}, {'emotion': 'disapproval', 'score': 0.053242359310388565}, {'emotion': 'gratitude', 'score': 0.017002204433083534}, {'emotion': 'amusement', 'score': 0.014057890512049198}]

Saved predictions to: /kaggle/working/roberta_utterance_only_model/te

,utterance,true_primary_emotion,predicted_labels,predicted_scores
0,Sebastian's injuries aren't as bad as I though...,surprise,"[joy, relief]","[0.5563510656356812, 0.9969531297683716]"
1,I wish I could feel human emotions like love o...,longing,"[desire, joy, optimism, sadness]","[0.8713105916976929, 0.4650634527206421, 0.802..."
2,I truly hope that this'mixing of minds' does m...,love,"[desire, optimism]","[0.7913300395011902, 0.9957618117332458]"
3,We failed to capture the French castle. We wer...,disappointment,"[annoyance, desire, disappointment, optimism]","[0.9763031005859375, 0.9689637422561646, 0.986..."
4,I can't believe those nihilists burned down th...,anger,"[anger, annoyance, disapproval, disgust, surpr...","[0.9966745376586914, 0.8954665660858154, 0.394..."
5,How dare that coach talk down to Eric like tha...,anger,"[admiration, anger, approval, caring, disappoi...","[0.7918128371238708, 0.96293705701828, 0.55790..."
6,I didn't expect Shevek to be such a passionate...,surprise,"[admiration, surprise]","[0.998499870300293, 0.9952291250228882]"
7,I'm so proud of my dad and his team. They've w...,love,"[admiration, approval, caring, joy, love, pride]","[0.6325715184211731, 0.5741339921951294, 0.531..."
8,Byam's acquittal? I can't believe it. The Brit...,surprise,"[disappointment, disapproval, surprise]","[0.6377963423728943, 0.9270638823509216, 0.998..."
9,I wonder what lies ahead on our journey to fin...,neutral,"[curiosity, excitement]","[0.997761607170105, 0.9875168204307556]"
